In [ ]:



from __future__ import annotations

import json
import os
import shutil
import subprocess
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.rdchem import RWMol


FLOAT_TOL = 1.0e-4


@dataclass
class QuantumSettings:
    

    g16_exe: str
    formchk_exe: str
    multiwfn_exe: str


@dataclass
class RepeatUnitResult:
    

    name: str
    smiles: str
    formal_charge: int
    multiplicity: int
    atom_count: int
    gjf_path: Path
    chk_path: Path
    out_path: Path
    fchk_path: Path
    chg_path: Path
    chgcons_path: Path
    atom_map_path: Path
    mapped_charges_path: Path


def load_quantum_settings() -> QuantumSettings:
    
    settings: Dict[str, Any] = {}
    try:
        from cemp_software_settings import load_and_apply_settings  

        settings = load_and_apply_settings()
    except Exception:
        settings = {}

    g16_exe = (
        settings.get("g16")
        or settings.get("gaussian16_exe")
        or shutil.which("g16")
        or "g16"
    )
    formchk_exe = (
        settings.get("gaussian16_formchk")
        or shutil.which("formchk")
        or "formchk"
    )
    multiwfn_exe = (
        settings.get("multiwfn_exe")
        or shutil.which("Multiwfn")
        or shutil.which("multiwfn")
        or "Multiwfn"
    )

    missing = [
        label
        for label, exe in (
            ("g16", g16_exe),
            ("formchk", formchk_exe),
            ("Multiwfn", multiwfn_exe),
        )
        if shutil.which(str(exe)) is None and not Path(str(exe)).is_file()
    ]
    if missing:
        raise FileNotFoundError("Public message removed for release.")

    return QuantumSettings(
        g16_exe=str(g16_exe),
        formchk_exe=str(formchk_exe),
        multiwfn_exe=str(multiwfn_exe),
    )


def _safe_name(name: str) -> str:
    
    return "".join(ch if ch.isalnum() or ch in "._+-" else "_" for ch in str(name))


def parse_charge_file(chg_path: Path) -> List[Dict[str, Any]]:
    
    rows: List[Dict[str, Any]] = []
    with chg_path.open("r", encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, 1):
            parts = line.split()
            if not parts:
                continue
            if len(parts) < 5:
                raise ValueError("Public message removed for release.")
            try:
                charge = float(parts[-1])
            except ValueError as exc:
                raise ValueError("Public message removed for release.") from exc
            rows.append(
                {
                    "index0": len(rows),
                    "index1": len(rows) + 1,
                    "element": parts[0],
                    "charge": charge,
                }
            )
    if not rows:
        raise ValueError("Public message removed for release.")
    return rows


def build_capped_repeat_unit(name: str, smiles: str) -> Tuple[Chem.Mol, Chem.Mol, List[Dict[str, Any]]]:
    
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        raise ValueError("Public message removed for release.")

    starred_mol_with_h = Chem.AddHs(mol)
    dummy_indices = [
        atom.GetIdx()
        for atom in starred_mol_with_h.GetAtoms()
        if atom.GetAtomicNum() == 0
    ]
    if len(dummy_indices) != 2:
        raise ValueError("Public message removed for release.")

    rw_mol = RWMol(starred_mol_with_h)
    dummy_index_set = set(dummy_indices)
    atom_map: List[Dict[str, Any]] = []

    for atom in starred_mol_with_h.GetAtoms():
        idx = atom.GetIdx()
        original_symbol = atom.GetSymbol()
        original_label = f"*{idx}_{name}" if original_symbol == "*" else f"{original_symbol}{idx}_{name}"
        atom_map.append(
            {
                "original_index0": idx,
                "gaussian_index0": idx,
                "gaussian_index1": idx + 1,
                "original_symbol": original_symbol,
                "gaussian_symbol": "H" if idx in dummy_index_set else original_symbol,
                "node_label": original_label,
                "is_dummy_replacement": idx in dummy_index_set,
                "formal_charge": int(atom.GetFormalCharge()),
                "neighbor_indices0": [nbr.GetIdx() for nbr in atom.GetNeighbors()],
            }
        )

    for idx in dummy_indices:
        rw_mol.ReplaceAtom(idx, Chem.Atom(1))
        rw_mol.GetAtomWithIdx(idx).SetNoImplicit(False)

    rw_mol.UpdatePropertyCache(strict=False)
    capped_mol_with_h = rw_mol.GetMol()
    Chem.SanitizeMol(capped_mol_with_h)

    return starred_mol_with_h, capped_mol_with_h, atom_map


def calculate_charge_and_multiplicity(mol: Chem.Mol) -> Tuple[int, int]:
    
    charge = int(sum(atom.GetFormalCharge() for atom in mol.GetAtoms()))
    multiplicity = int(sum(atom.GetNumRadicalElectrons() for atom in mol.GetAtoms()) + 1)
    return charge, multiplicity


def write_gaussian_input(
    capped_mol: Chem.Mol,
    name: str,
    charge: int,
    multiplicity: int,
    output_dir: Path,
    nproc: int = 10,
    mem_gb: int = 32,
) -> Tuple[Path, Path, Path]:
    
    output_dir.mkdir(parents=True, exist_ok=True)
    work_mol = Chem.Mol(capped_mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 20260428
    status = AllChem.EmbedMolecule(work_mol, params)
    if status != 0:
        status = AllChem.EmbedMolecule(work_mol, useRandomCoords=True, randomSeed=20260428)
    if status != 0:
        raise RuntimeError("Public message removed for release.")

    try:
        AllChem.MMFFOptimizeMolecule(work_mol, mmffVariant="MMFF94")
    except Exception:
        AllChem.UFFOptimizeMolecule(work_mol)

    safe = _safe_name(name)
    gjf_path = output_dir / f"{safe}.gjf"
    chk_path = output_dir / f"{safe}.chk"
    out_path = output_dir / f"{safe}.out"

    xyz_lines = Chem.MolToXYZBlock(work_mol).splitlines()[2:]
    with gjf_path.open("w", encoding="utf-8", newline="\n") as handle:
        handle.write(f"%nprocshared={nproc}\n")
        handle.write(f"%mem={mem_gb}GB\n")
        handle.write(f"%chk={chk_path.resolve()}\n")
        handle.write("# opt freq b3lyp/6-311g(d,p) em=gd3bj scale=0.9682\n\n")
        handle.write(f"{safe}.gjf\n\n")
        handle.write(f"{charge} {multiplicity}\n")
        handle.write("\n".join(xyz_lines))
        handle.write("\n\n")

    return gjf_path, chk_path, out_path


def write_atom_map(atom_map: List[Dict[str, Any]], path: Path) -> None:
    
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(atom_map, handle, ensure_ascii=False, indent=2)


def write_chgcons(atom_map: List[Dict[str, Any]], formal_charge: int, path: Path) -> None:
    
    constrained_indices = [
        int(item["gaussian_index1"])
        for item in atom_map
        if not bool(item["is_dummy_replacement"])
    ]
    if not constrained_indices:
        raise ValueError("Public message removed for release.")
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="\n") as handle:
        handle.write(",".join(str(idx) for idx in constrained_indices))
        handle.write(f" {int(formal_charge)}")


def run_gaussian(gjf_path: Path, out_path: Path, settings: QuantumSettings) -> None:
    
    cmd = [settings.g16_exe, str(gjf_path), str(out_path)]
    subprocess.run(cmd, check=True)
    content = out_path.read_text(encoding="utf-8", errors="ignore")
    if "Normal termination" not in content:
        raise RuntimeError("Public message removed for release.")


def run_formchk(chk_path: Path, fchk_path: Path, settings: QuantumSettings) -> None:
    
    subprocess.run([settings.formchk_exe, str(chk_path), str(fchk_path)], check=True)


def run_multiwfn_resp(fchk_path: Path, chgcons_path: Path, settings: QuantumSettings, cwd: Path) -> None:
    
    commands = f"7\n18\n6\n1\n{chgcons_path.name}\n2\ny\n0\n0\nq"
    process = subprocess.Popen(
        [settings.multiwfn_exe, str(fchk_path.resolve())],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        cwd=str(cwd),
        text=True,
    )
    stdout, stderr = process.communicate(commands)
    (cwd / f"{fchk_path.stem}_Multiwfn_stdout.txt").write_text(stdout or "", encoding="utf-8")
    (cwd / f"{fchk_path.stem}_Multiwfn_stderr.txt").write_text(stderr or "", encoding="utf-8")
    if process.returncode not in (0, None):
        raise RuntimeError("Public message removed for release.")


def create_mapped_charges(
    name: str,
    atom_map: List[Dict[str, Any]],
    chg_path: Path,
    output_path: Path,
    formal_charge: int,
) -> Dict[str, Dict[str, Any]]:
    
    chg_rows = parse_charge_file(chg_path)
    if len(chg_rows) != len(atom_map):
        raise ValueError(
            "Public message removed for release."
        )

    mapped: Dict[str, Dict[str, Any]] = {}
    non_dummy_sum = 0.0
    total_sum = 0.0
    for item, row in zip(atom_map, chg_rows):
        expected_element = str(item["gaussian_symbol"])
        if row["element"] != expected_element:
            raise ValueError(
                "Public message removed for release."
                f".chg={row['element']} map={expected_element}"
            )
        charge = float(row["charge"])
        total_sum += charge
        if not bool(item["is_dummy_replacement"]):
            non_dummy_sum += charge
        mapped[str(item["node_label"])] = {
            "charge": charge,
            "original_index0": int(item["original_index0"]),
            "gaussian_index0": int(item["gaussian_index0"]),
            "gaussian_index1": int(item["gaussian_index1"]),
            "original_symbol": str(item["original_symbol"]),
            "gaussian_symbol": expected_element,
            "is_dummy_replacement": bool(item["is_dummy_replacement"]),
            "formal_charge": int(item["formal_charge"]),
        }

    if abs(non_dummy_sum - float(formal_charge)) > FLOAT_TOL:
        raise ValueError(
            "Public message removed for release."
        )

    payload = {
        "name": name,
        "formal_charge": int(formal_charge),
        "total_chg_sum": total_sum,
        "non_dummy_charge_sum": non_dummy_sum,
        "mapped_charges": mapped,
    }
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)
    return mapped


def prepare_repeat_unit_resp(
    name: str,
    smiles: str,
    base_dir: Path,
    settings: Optional[QuantumSettings] = None,
    force_recompute: bool = False,
    nproc: int = 10,
    mem_gb: int = 32,
) -> RepeatUnitResult:
    
    base_dir = Path(base_dir)
    safe = _safe_name(name)
    gaussian_dir = base_dir / "Gaussian" / "RESPpolymer" / "success"
    mapped_dir = base_dir / "mapped_charges"
    atom_map_path = mapped_dir / f"{safe}_resp_atom_map.json"
    mapped_charges_path = mapped_dir / f"{safe}_mapped_charges.json"
    chgcons_path = base_dir / f"{safe}_chgcons.txt"
    fchk_path = base_dir / f"{safe}.fchk"
    chg_path = base_dir / f"{safe}.chg"

    starred_mol, capped_mol, atom_map = build_capped_repeat_unit(name, smiles)
    formal_charge, _ = calculate_charge_and_multiplicity(starred_mol)
    capped_charge, multiplicity = calculate_charge_and_multiplicity(capped_mol)
    if formal_charge != capped_charge:
        raise ValueError(
            "Public message removed for release."
        )

    write_atom_map(atom_map, atom_map_path)
    write_chgcons(atom_map, formal_charge, chgcons_path)

    gjf_path = gaussian_dir / f"{safe}.gjf"
    chk_path = gaussian_dir / f"{safe}.chk"
    out_path = gaussian_dir / f"{safe}.out"

    should_run_quantum = force_recompute or not (fchk_path.exists() and chg_path.exists())
    if should_run_quantum:
        settings = settings or load_quantum_settings()
        gjf_path, chk_path, out_path = write_gaussian_input(
            capped_mol=capped_mol,
            name=name,
            charge=formal_charge,
            multiplicity=multiplicity,
            output_dir=gaussian_dir,
            nproc=nproc,
            mem_gb=mem_gb,
        )
        run_gaussian(gjf_path, out_path, settings)
        run_formchk(chk_path, fchk_path, settings)
        run_multiwfn_resp(fchk_path, chgcons_path, settings, cwd=base_dir)

    if not chg_path.exists():
        raise FileNotFoundError("Public message removed for release.")
    create_mapped_charges(name, atom_map, chg_path, mapped_charges_path, formal_charge)

    return RepeatUnitResult(
        name=name,
        smiles=smiles,
        formal_charge=formal_charge,
        multiplicity=multiplicity,
        atom_count=len(atom_map),
        gjf_path=gjf_path,
        chk_path=chk_path,
        out_path=out_path,
        fchk_path=fchk_path,
        chg_path=chg_path,
        chgcons_path=chgcons_path,
        atom_map_path=atom_map_path,
        mapped_charges_path=mapped_charges_path,
    )


def load_mapped_charge_payload(path: Path) -> Dict[str, Any]:
    
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)



In [ ]:

EXCEL_FILE = "System_random_copolymer.xlsx"
FORCE_RECOMPUTE_RESP = False  
NPROC = 10
MEM_GB = 32

import pandas as pd
from pathlib import Path


def read_polymer_rows(excel_path: Path) -> pd.DataFrame:
    
    df = pd.read_excel(excel_path)
    missing = {"Name", "SMILES"}.difference(df.columns)
    if missing:
        raise ValueError("Public message removed for release.")

    work_df = df.dropna(subset=["Name", "SMILES"]).copy()
    work_df["Name"] = work_df["Name"].astype(str).str.strip()
    work_df["SMILES"] = work_df["SMILES"].astype(str).str.strip()
    work_df = work_df[
        (work_df["Name"] != "")
        & (work_df["SMILES"] != "")
        & (work_df["Name"].str.lower() != "nan")
        & (work_df["SMILES"].str.lower() != "nan")
    ].copy()

    if "is polymer" in work_df.columns:
        polymer_df = work_df[work_df["is polymer"].astype(bool)].copy()
    else:
        polymer_df = work_df.copy()

    if polymer_df.empty:
        raise ValueError("Public message removed for release.")
    if polymer_df["Name"].astype(str).duplicated().any():
        duplicated = polymer_df.loc[polymer_df["Name"].astype(str).duplicated(), "Name"].astype(str).tolist()
        raise ValueError("Public message removed for release.")
    return polymer_df


base_dir = Path.cwd()
polymer_df = read_polymer_rows(Path(EXCEL_FILE))


for row in polymer_df.itertuples(index=False):
    name = str(getattr(row, "Name"))
    smiles = str(getattr(row, "SMILES"))
    result = prepare_repeat_unit_resp(
        name=name,
        smiles=smiles,
        base_dir=base_dir,
        settings=None,
        force_recompute=FORCE_RECOMPUTE_RESP,
        nproc=NPROC,
        mem_gb=MEM_GB,
    )
    print(f"[OK] {name}: formal_charge={result.formal_charge}, atoms={result.atom_count}")
    print(f"     atom_map={result.atom_map_path}")
    print(f"     mapped_charges={result.mapped_charges_path}")
